# Диагностика support дозовой модели

Ноутбук **не переоценивает** статическую дозовую DiD и dose event-study.
Он повторяет определения `R_h`, `Δw_h`, `W_h^+`, `W_h^-` и правила выборки
из `final_empirical_recalculation.ipynb`, затем считает support для
`R_h × Δw_h × change_type` на уровне гексагонов и hex-day.

Коэффициенты `β_R`, `β_+`, `β_-` — условные компоненты одной аддитивной
спецификации, а не категориальные ATT `region_only` / `workmode_only`.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from run_dose_support_diagnostics import run

OUT = PROJECT_ROOT / "outputs" / "final"
APPS = PROJECT_ROOT / "data" / "raw" / "application_dataset.csv"
HEXS = PROJECT_ROOT / "data" / "raw" / "hexagons_dataset.csv"

tables = run(OUT, APPS, HEXS)
print("Primary estimation sample support (sch_flg / outcome_specific):")
display(tables["primary"])
print("Identification flags:")
display(tables["notes"])
print("Overlap R=0 vs R=1 by delta_w (sch_flg / outcome_specific):")
display(
    tables["overlap"][
        (tables["overlap"]["outcome"] == "sch_flg")
        & (tables["overlap"]["cohort_specification"] == "outcome_specific")
    ]
)


## Определения (как в коде дозовой модели)

- `R_h = I(region_id_old != region_id_new)`
- `delta_w_h = work_mode_new - work_mode_old`
- `W_h_plus = max(delta_w_h, 0)` — число добавленных рабочих дней
- `W_h_minus = max(-delta_w_h, 0)` — положительное число убранных дней
- выборка: pooled CORE treated + never-treated; когорта `2022-10-19` исключена
- regressors: `post_R`, `post_W_plus`, `post_W_minus`

Файлы:
- `dose_support_R_by_delta.csv` — exact hex / hex-day support на estimation sample
- `dose_support_R_delta_overlap.csv` — перекрытие `R=0/1` по `Δw`
- `dose_support_identification_flags.csv` — флаги слабого overlap


## Вывод

Ноутбук предназначен для диагностики support дозовой спецификации
по сочетаниям `R_h × Δw_h × change_type` на уровне гексагонов и
hex–day без переоценки статической DiD и dose event-study.

Диагностика проверяет, достаточно ли перекрытия значений `Δw_h` при
`R_h = 0` и `R_h = 1` для раздельной интерпретации условных
компонентов `β_R`, `β_+` и `β_-` внутри одной аддитивной модели.
Таблицы сохраняются в `outputs/final/dose_support_*.csv`.

По рассчитанным таблицам support частично перекрыт: ненулевые
`Δw_h ∈ {-5,-4,-3,-2,-1,1}` встречаются и при `R_h=0`, и при `R_h=1`,
тогда как крупные положительные дозы и часть крайних отрицательных
значений сосредоточены преимущественно в одной группе
(`region_and_workmode` или `workmode_only`). Это не отменяет модель,
но ограничивает линейную dose–response интерпретацию на всём
диапазоне `Δw_h` и сохраняет статус дозовых коэффициентов как
условных компонентов robustness-спецификации, а не категориальных ATT.
